# Head model

We build an individual head model using a boundary element model (BEM). A BEM consists of all surfaces between the sources and the electrodes. They include the brain surface, inner skull, outer skull and skin. <br>
The brain surface was already computed during the FreeSurfer step. Here we compute the skull and skin surfaces.

## Imports

In [1]:
import mindmap as mm
import mne
from pathlib import Path
import pyvista as pv
import nibabel as nib
import numpy as np

## Data

In [2]:
sub     = "sub-pat10"
dataset = "spike"

# configuration
config = mm.io.read_config("../configurations/spike.yml")
fs_dir = Path(config["FS_DIR"])
mm_dir = Path(config["MM_DIR"])

# files
#bem_folder = fs_dir/sub/"bem"
#if not bem_folder.exists():
#    bem_folder.mkdir()
eeg     = f"/home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/cartool-v3.80/{sub}/ses-preop/eeg"
sources = mm_dir/sub/"src"/f"{sub}_ddm-surf-7-src.fif"

# file for visualization
mri = fs_dir/sub/"mri"/"T1.mgz"

# parameters
conductivities=[0.3, 0.006, 0.3]
min_skull_thick=2

# plotting parameters
plot_bem_kwargs = dict(
    subject=sub,
    subjects_dir=fs_dir,
    brain_surfaces="white",
    orientation="coronal",
    slices=[50, 100, 150, 200]
)

# load
mri = nib.load(mri)
eeg = mm.io.read_eeg(eeg, dataset)

## Create Surfaces from T1-weighted MRI
Run time: 5 min <br>
We use a watershed algorithm (2001).

In [3]:
mne.bem.make_watershed_bem(
    subject = sub,
    subjects_dir = fs_dir,
    volume = "T1",
    overwrite = True,
)


Running mri_watershed for BEM segmentation with the following parameters:

Results dir = /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/bem/watershed
Command = mri_watershed -useSRAS -surf /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/bem/watershed/sub-pat10 /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/mri/T1.mgz /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/bem/watershed/ws.mgz

Running subprocess: mri_watershed -useSRAS -surf /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/bem/watershed/sub-pat10 /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/mri/T1.mgz /home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/freesurfer-7.4.1/sub-pat10/bem/watershed/ws.mgz

Mode:          use surfaceRAS to save surface vertex positions
Mode:          Saving out BEM surfaces

**********

#### Visualize Surfaces

In [4]:
names    = ["outer_skin.surf", "outer_skull.surf", "inner_skull.surf"]
images   = []
for n in names:
    rr, tri = nib.freesurfer.read_geometry(fs_dir/sub/"bem"/n)
    rr = nib.affines.apply_affine(np.linalg.inv(mri.affine), rr).astype(int)

    vol = np.zeros(mri.shape)
    vol[rr[:,0], rr[:,1], rr[:,2]] = 255

    img = pv.ImageData(dimensions=mri.shape, spacing=(1,1,1), origin=(0,0,0))
    img.point_data["scalars"] = vol.flatten(order="F")

    images.append(img)


volume = pv.ImageData(dimensions=mri.shape, spacing=(1,1,1), origin=(0,0,0))
volume.point_data["scalars"] = mri.get_fdata().flatten(order="F")

plotter = pv.Plotter()
plotter.add_mesh(volume.contour(3), color="white", opacity=0.05)

def create_slice(normal, origin):
    slc_mri    = volume.slice(normal=normal, origin=origin)
    slc_skin   = images[0].slice(normal=normal, origin=origin)
    slc_oskull = images[1].slice(normal=normal, origin=origin)
    slc_iskull = images[2].slice(normal=normal, origin=origin)
    plotter.add_mesh(slc_mri, cmap="bone", name="mri", opacity="linear")
    plotter.add_mesh(slc_skin, cmap="Reds", name="skin", opacity="linear")
    plotter.add_mesh(slc_oskull, cmap="Reds", name="oskull", opacity="linear")
    plotter.add_mesh(slc_iskull, cmap="Greens", name="iskull", opacity="linear")


plotter.add_plane_widget(
    create_slice,
)
plotter.show_grid()
plotter.show_axes()
plotter.show()

Widget(value='<iframe src="http://localhost:40649/index.html?ui=P_0x7f7cb8499d30_0&reconnect=auto" class="pyvi…

#### Correct Surfaces
The waterhsed algorithm sometimes generates an inner skull that goes outside the outer skull. We implemented some functions to correct for this.

In [5]:
from mindmap.utils.mri import correct_skull 

correct_skull(
    sub,
    offset = 2,
    fs_dir = fs_dir,
    save_arrows = True,
)

# Correct skull sends an "ERROR:root:No points/cells to operate on" when there are no 
# correction to be made. The program will continue without stopping.

ERROR:root:No points/cells to operate on
ERROR:root:No points/cells to operate on
2024-10-18 08:52:08.712 ( 114.913s) [    7F7CBB7E2740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
2024-10-18 08:52:08.713 ( 114.914s) [    7F7CBB7E2740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x557e0126dfd0): No points/cells to operate on
2024-10-18 08:52:08.713 ( 114.914s) [    7F7CBB7E2740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x557e0126dfd0): No points/cells to operate on


#### Create Boudary Element Model (BEM)
In this section we apply some topological checks to verify if the surfaces are compatible with eachother in order to later make a conductor model.

In [6]:
head_model = mne.make_bem_model(
        subject=sub,
        subjects_dir=fs_dir,
        conductivity=conductivities,
        #ico=None,
)

Creating the BEM geometry...
Going from 5th to 4th subdivision of an icosahedron (n_tri: 20480 -> 5120)
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
Going from 5th to 4th subdivision of an icosahedron (n_tri: 20480 -> 5120)
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
Going from 5th to 4th subdivision of an icosahedron (n_tri: 20480 -> 5120)
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
outer skin  CM is   1.26 -16.25   4.74 mm
outer skull CM is   1.27 -16.10   4.24 mm
inner skull CM is   0.96 -18.89  13.02 mm
Checking that surface outer skull is inside surface outer skin  ...
Checking that surface inner skull is inside surface outer skull ...
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
Checking distance between outer skin  and outer skull surfaces...
Minimum distance between the outer skin  and outer skull surfaces is approxima

## Register Electrodes

In [7]:
points = mm.utils.get_ch_positions(eeg)
elec   = pv.PolyData(points)
head   = mm.utils.surf_to_polydata(head_model[0]["rr"], head_model[0]["tris"])

plotter = pv.Plotter()
plotter.add_mesh(elec, render_points_as_spheres=True, color="red", name="initial electrodes")
plotter.add_mesh(head, smooth_shading=True, color="lightgray", name="head")
plotter.show()

Widget(value='<iframe src="http://localhost:40649/index.html?ui=P_0x7f7b19b56090_1&reconnect=auto" class="pyvi…

#### Iterative Closest Point (ICP)

#### Projection

In [8]:
points = mm.utils.project_points_to_surface(points, head)

In [9]:
elec_new = pv.PolyData(points)
_ = plotter.add_mesh(elec_new, render_points_as_spheres=True, color="green", name="new electrodes")

#### Update new electrode positions

In [10]:
data     = dict(zip(eeg.info["ch_names"], points))
montage  = mne.channels.make_dig_montage(ch_pos=data, coord_frame="head")
eeg      = eeg.set_montage(montage)

## Solve Forward Problem

In [11]:
bem = mne.make_bem_solution(head_model)
trans = mne.transforms.Transform(fro="head", to="mri")      # identity matrix

forward = mne.make_forward_solution(
    info=eeg.info,
    trans=trans,
    src=sources,
    bem=bem,
    meg=False,
    eeg=True,
    verbose=False,
)

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (2562) -> outer skin  (2562) ...
        outer skin  (2562) -> outer skull (2562) ...
        outer skin  (2562) -> inner skull (2562) ...
        outer skull (2562) -> outer skin  (2562) ...
        outer skull (2562) -> outer skull (2562) ...
        outer skull (2562) -> inner skull (2562) ...
        inner skull (2562) -> outer skin  (2562) ...
        inner skull (2562) -> outer skull (2562) ...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


#### Visualize Forward solution

In [13]:
mm.viz.plot_leadfield(eeg, forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:40649/index.html?ui=P_0x7f7acc5fede0_2&reconnect=auto" class="pyvi…

## Save Forward Solution

In [15]:
# save source space
folder_fwd = mm_dir/sub/"fwd"
folder_eeg = mm_dir/sub/"eeg"

if folder_fwd.exists() == False:
        folder_fwd.mkdir(parents=True)
if folder_eeg.exists() == False:
        folder_eeg.mkdir(parents=True)

print(f"Saving head model and forward solution for subject: {sub}")
mne.write_bem_surfaces(folder_fwd/f"{sub}_hm-bem.fif", head_model, overwrite=True)
mne.write_forward_solution(folder_fwd/f"{sub}_bem-fwd.fif", forward, overwrite=True)

print(f"Saving co-registered eeg for subject: {sub}")
if dataset == "spike":
        eeg.save(folder_eeg/f"{sub}_coreg_ave.fif")
else:
        raise ValueError(f"Unknown dataset: {dataset}")

Saving head model and forward solution for subject: sub-pat10
Overwriting existing file.
    Write a source space...
    [done]
    Write a source space...
    [done]
    2 source spaces written
Saving co-registered eeg for subject: sub-pat10
